# Lecture Bot

Ask questions about the documents in `docs/` (placeholders for now: syllabus and module
description).

`MODE` decides how the documents reach the model:

- **`"stuffing"`** (context stuffing): every question is sent together with all documents.
  Misses nothing, but everything must fit into the context window.
- **`"rag"`** (Retrieval-Augmented Generation): documents are split into chunks, a search
  picks the chunks most similar to the question, and only those are sent. Scales to large
  collections, but the search can miss the right passage.

In [ ]:
from pathlib import Path
import math
import subprocess

import ollama

MODE = "stuffing"               # "stuffing" or "rag"
MODEL = "qwen2.5:7b"
EMBED_MODEL = "nomic-embed-text"
NUM_CTX = 8192
TOP_K = 3                        # rag only: chunks sent per question
DOCS_DIR = Path("docs")

<details>
<summary><b>Settings explained</b></summary>

- `MODEL`: the language model that answers.
- `EMBED_MODEL`: turns text into a vector (a list of numbers). Similar meaning gives similar
  vectors, which is what the RAG search compares.
- `NUM_CTX`: the context window, in tokens (word pieces, about 4 characters each).
  Instructions, documents, chat history and answer must all fit. Ollama cuts off the excess
  without an error.
- `TOP_K`: how many chunks RAG sends per question.
</details>

## 1. Load the documents

In [ ]:
def pdf_to_text(path):
    return subprocess.run(["pdftotext", "-layout", str(path), "-"],
                          capture_output=True, text=True, check=True).stdout

docs = {p.name: pdf_to_text(p) for p in sorted(DOCS_DIR.glob("*.pdf"))}

for name, text in docs.items():
    print(f"{name:40s} ~{len(text) // 4:5d} tokens")
print(f"{'total':40s} ~{sum(len(t) for t in docs.values()) // 4:5d} tokens, window {NUM_CTX}")

<details>
<summary><b>What happens here</b></summary>

- Models read text, not PDFs. `pdftotext` extracts the text; images and layout are lost.
- Tokens are estimated as characters divided by 4. The exact count appears after each answer.
- In stuffing mode, the total must stay well below the window.
</details>

## 2. Prepare the context

In [ ]:
def chunk(text, size=150, overlap=30):
    words = text.split()
    step = size - overlap
    return [" ".join(words[i:i + size]) for i in range(0, max(len(words) - overlap, 1), step)]

def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

if MODE == "rag":
    chunks = [(name, c) for name, text in docs.items() for c in chunk(text)]
    # nomic-embed-text was trained with these prefixes:
    vectors = ollama.embed(model=EMBED_MODEL,
                           input=["search_document: " + c for _, c in chunks]).embeddings
    print(f"{len(chunks)} chunks embedded")

def context_for(question):
    if MODE == "stuffing":
        return "\n\n".join(f"=== {name} ===\n{text}" for name, text in docs.items())
    q = ollama.embed(model=EMBED_MODEL, input="search_query: " + question).embeddings[0]
    ranked = sorted(zip(vectors, chunks), key=lambda vc: cosine(q, vc[0]), reverse=True)
    return "\n\n".join(f"=== {name} ===\n{c}" for _, (name, c) in ranked[:TOP_K])

<details>
<summary><b>How the two modes work</b></summary>

- **Stuffing:** `context_for()` returns all documents, whatever the question.
- **RAG:**
  1. `chunk()` splits each document into 150-word pieces. Neighbours overlap by 30 words, so
     a sentence cut at one border is whole in the next chunk.
  2. Every chunk is embedded once, up front.
  3. Per question, the question is embedded too, and `cosine()` scores each chunk: 1 means
     same direction (similar meaning), 0 means unrelated.
  4. The `TOP_K` best chunks become the context.
- `=== name ===` marks where each document starts, so the model can cite it.
</details>

## 3. Ask

In [ ]:
INSTRUCTIONS = """You are a teaching assistant for the module "Agentic AI".
Answer only from the lecture material below. If the material does not cover the
question, say so. Name the document you used. Answer in the language of the question."""

history = []

def reset():
    history.clear()

def ask(question):
    global last_prompt
    last_prompt = INSTRUCTIONS + "\n\n" + context_for(question)
    history.append({"role": "user", "content": question})
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "system", "content": last_prompt}] + history,
        options={"num_ctx": NUM_CTX, "temperature": 0},
    )
    answer = response.message.content
    history.append({"role": "assistant", "content": answer})
    print(answer)
    print(f"\n[{MODE}: {response.prompt_eval_count} of {NUM_CTX} tokens]")

<details>
<summary><b>Prompt and memory</b></summary>

- The **system message** holds the rules plus the context. It is rebuilt for every question.
- The instructions each have a job:
  - *Answer only from the material*: prevents *hallucination*, confidently invented answers.
  - *If not covered, say so*: gives the model a permitted way out.
  - *Name the document*: makes the answer checkable.
- The model has **no memory**. `history` resends all earlier questions and answers with every
  call, so each call gets longer. Call `reset()` when the token count nears the limit.
- In RAG mode, a follow-up such as "and after that?" retrieves poorly: the search sees only
  that one question.
- `temperature: 0` always picks the most likely token, so answers are reproducible.
- `print(last_prompt)` shows exactly what the model received for the last question.
</details>

In [ ]:
ask("Wann wird MCP behandelt und worum geht es dabei?")

In [ ]:
ask("Und in welcher Woche kommen danach die Reasoning Models?")

In [ ]:
ask("Wie heißt der Hund des Professors?")

<details>
<summary><b>What the three questions test</b></summary>

1. A lookup: the answer is in the syllabus.
2. A follow-up: only works if the history is sent along.
3. Not in the material: the bot should say so, not invent a name.
</details>

In [ ]:
# See exactly what the model received for the last question:
print(last_prompt)

In [ ]:
# Your questions:

<details>
<summary><b>Which mode, and is this state of the art?</b></summary>

- **Documents fit into the window:** use stuffing. For small collections it is the
  recommended default; Anthropic, for example, advises skipping RAG below about 200,000
  tokens.
- **Documents do not fit:** use RAG. Production systems add keyword search (BM25) and a
  reranking model on top of the vector search shown here.
- **Agentic RAG:** the model gets search as a tool and decides itself what to look up (the
  loop from Lab 02 with the tools from Lab 03). Lab 09 covers RAG in depth.
- The window here is 8,192 tokens (about 12 pages), because everyone shares one server.
  Commercial models accept hundreds of thousands.
- *Prompt injection* is not another word for stuffing: it names an attack where text inside
  a document overrides the instructions (Lab 13).
</details>